# ML-00 ML Input Export from 03 Feature Output

`data/preprocessing/03_ml_feature_process.ipynb`가 만든 `ml_exp00.parquet` 단일 산출물을 사용해 `ml_*.py` 모듈에 넣을 수 있는 입력 파일 세트를 생성합니다.

이 노트북은 **학습/검증/test를 실행하지 않습니다.** `fb_*.py` 모듈까지만 사용해 입력 산출물 생성 단계에서 멈춥니다.

생성 목표:

1. `ml_exp00.parquet` 단일 파일의 schema와 `split` 컬럼 확인
2. `ml_feature_columns.csv`의 `used_in_ml=True` feature 목록 확인
3. 단일 parquet를 `split` 컬럼 기준으로 train/val/test parquet 3개로 분리
4. `ml_feature_columns.csv`를 같은 output directory에 복사
5. 후속 `ml_train.py`, `ml_val.py`, `ml_test.py`에 넘길 경로를 manifest로 저장

## 실행 범위

포함:

- `fb_utils.py` 기준 프로젝트 경로 설정
- `fb_build.split_single_parquet_by_existing_split()` 실행
- ML 입력 파일 존재 여부와 schema 확인
- `ml_inputs_manifest.json` 저장

제외:

- `ml_io.py`, `ml_train.py`, `ml_val.py`, `ml_test.py` import
- XGBoost 학습
- validation threshold 선택
- final test 평가
- metric 생성

## 실행 전 주의

- 원본 `data/processed/ml_features/ml_exp00.parquet`는 수정하지 않습니다.
- 원본 `ml_feature_columns.csv`는 수정하지 않고 output directory로 복사합니다.
- split parquet는 `ml/ml-00_baseline/outputs/feature_build/` 아래 새 run directory에 저장합니다.
- 기본 overwrite는 `False`입니다. 같은 설정으로 재실행하려면 `RUN_ID`를 바꾸는 것을 우선 권장합니다.
- 이 노트북이 만든 parquet/model/manifest 산출물은 commit 대상이 아닙니다.

In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# 0. Project root, local module path, and feature-build utilities
# ============================================================
# 이 노트북은 feature build/export 전용입니다.
# 학습/검증/test 모듈(ml_io, ml_train, ml_val, ml_test)은 import하지 않습니다.
#
# Jupyter의 Path.cwd()는 노트북 파일 위치가 아니라 커널 작업 디렉터리입니다.
# 따라서 cwd에 의존하지 않고 Git repository root를 먼저 계산합니다.
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True,
    ).strip()
).resolve()

# fb_*.py 모듈은 ml/ml-00_baseline/feature_build 아래에 있습니다.
# 상위 폴더명에 하이픈이 포함되어 일반 package import가 불편하므로,
# 모듈 폴더를 sys.path에 추가한 뒤 flat import로 사용합니다.
MODULE_DIR = PROJECT_ROOT / "ml" / "ml-00_baseline" / "feature_build"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# feature build/export 단계에서 필요한 모듈만 import합니다.
# fb_utils는 Git root 기준 공통 경로와 seed 고정 함수를 제공합니다.
import fb_utils
from fb_build import split_single_parquet_by_existing_split
from fb_schema import validate_no_forbidden_feature_columns

SEED = 42
fb_utils.set_seed(SEED, use_torch=False)

# 경로 상수는 fb_utils가 계산한 Git root 기준 값을 사용합니다.
BASE_DIR = fb_utils.BASE_DIR
PROCESSED_DIR = fb_utils.PROCESSED_DIR
PROJECT_ROOT = BASE_DIR
FB_UTILS_PATH = Path(fb_utils.__file__).resolve()

print("PROJECT_ROOT :", PROJECT_ROOT)
print("MODULE_DIR   :", MODULE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("fb_utils.py  :", FB_UTILS_PATH)

## 1. 실행 설정

아래 셀만 수정해서 입력 산출물 위치, output directory, overwrite 정책을 제어합니다.

이 노트북은 full split parquet를 만드는 산출물 생성 단계입니다. sample 학습 옵션은 없습니다.

In [ ]:
# ------------------------------------------------------------
# 1. User-editable settings
# ------------------------------------------------------------
# 입력 파일과 출력 위치를 한 곳에서 관리하는 설정 셀
#
# 이 셀에서 관리하는 값:
# - 전처리 노트북이 만든 단일 ML feature parquet 원본 경로
# - 전처리 노트북이 만든 feature catalog CSV 원본 경로
# - 후속 ml_train.py, ml_val.py, ml_test.py에 넘길 train/val/test split parquet 생성 경로
# - ML 학습에 실제 사용할 feature catalog CSV 사본 경로
# - 기존 산출물 overwrite 정책
#
# 주의:
# - INPUT_SINGLE_PARQUET 원본 파일은 수정금지 (전처리 노트북 결과임)
# - FEATURE_COLUMNS_SOURCE_PATH 원본 파일은 수정금지 (전처리 노트북 결과임)
# - FEATURE_COLUMNS_EXPORT_PATH가 ML 학습에 들어갈 최종 feature catalog임
# - EXPORT_OUTPUT_DIR 아래에 후속 ML 입력 후보 파일을 새로 생성하는 형태로 사용
# - 검토 후 확정 입력으로 사용할 파일은 사용자가 ml/ml-00_baseline/inputs/ 아래로 직접 옮겨야 함

# ------------------------------------------------------------
# 1.1 Experiment and run identifiers
# ------------------------------------------------------------
# SOURCE_EXPERIMENT_ID: 입력 데이터셋 파일명을 지정하는 ID
# 예: ml_exp00 -> data/processed/ml_features/ml_exp00.parquet
SOURCE_EXPERIMENT_ID = "ml_exp00"

# EXPORT_EXPERIMENT_ID: split export 결과 파일명 prefix와 output directory 이름에 사용될 문구
EXPORT_EXPERIMENT_ID = "ml_exp00_test_run"

# EXPORT_NAME: outputs/feature_build 경로에 생성될 산출물 폴더 이름
EXPORT_NAME = "ml-00-ml-input-export"

# RUN_ID: 같은 export를 여러 번 실행할 때 결과가 섞이지 않도록 구분하는 실행 ID
# 재실행 시 overwrite=True보다 run_001, run_002처럼 RUN_ID 변경 권장
RUN_ID = "run_000"

# 정답 라벨 컬럼명
LABEL_COL = "label"

# ------------------------------------------------------------
# 1.2 Source input paths
# ------------------------------------------------------------
# 전처리 노트북 ML feature 산출물이 저장된 기본 폴더
# 직접 지정 필요 시 아래 값을 BASE_DIR 기준 경로로 수정 가능
SOURCE_ML_FEATURE_DIR = PROCESSED_DIR / "ml_features"

# 전처리 노트북이 만든 단일 ML feature parquet 원본 경로
# split 컬럼을 포함해야 하며, 이 원본 파일은 수정하지 않음
INPUT_SINGLE_PARQUET = SOURCE_ML_FEATURE_DIR / f"{SOURCE_EXPERIMENT_ID}.parquet"

# 전처리 노트북이 만든 feature catalog CSV 원본 경로
# 이 파일은 원본이므로 직접 수정하지 않음
# ML 파트에서는 이 파일을 EXPORT_OUTPUT_DIR로 복사한 뒤 used_in_ml 컬럼을 검토/수정한 사본을 사용
FEATURE_COLUMNS_SOURCE_PATH = SOURCE_ML_FEATURE_DIR / "ml_feature_columns.csv"


# ------------------------------------------------------------
# 1.3 Export output paths
# ------------------------------------------------------------
# 후속 ml_*.py에 넘길 산출물을 feature_build output 폴더에 생성
EXPORT_OUTPUT_DIR = (
    BASE_DIR
    / "ml"
    / "ml-00_baseline"
    / "outputs"
    / "feature_build"
    / EXPORT_NAME
    / EXPORT_EXPERIMENT_ID
    / RUN_ID
)

# 모든 export 산출물 파일명에 export 작업 정보를 포함합니다.
# 이 값은 split_single_parquet_by_existing_split(..., experiment_id=ARTIFACT_PREFIX)에 그대로 전달됩니다.
# fb_build는 {experiment_id}_Xy_train.parquet 규칙으로 파일을 만들기 때문에 아래 경로도 같은 규칙으로 맞춥니다.
ARTIFACT_PREFIX = f"{EXPORT_NAME}__{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

# split export 결과 파일 경로입니다.
# 후속 ml_train.py, ml_val.py, ml_test.py에 넘길 parquet 입력 후보입니다.
TRAIN_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
VAL_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
TEST_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"

# fb_build가 함께 저장하는 split-only 실행 요약 파일 경로입니다.
SPLIT_SUMMARY_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_summary.csv"
SPLIT_BUILD_SUMMARY_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_existing_summary.json"

# ML 학습에 실제 사용할 feature catalog CSV 사본 경로입니다.
# 원본 FEATURE_COLUMNS_SOURCE_PATH는 수정하지 않고, 이 사본을 검토/수정한 뒤 feature_columns_path로 전달합니다.
FEATURE_COLUMNS_EXPORT_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_ml_feature_columns.csv"

# 이 노트북 자체의 export manifest 경로입니다.
MANIFEST_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_ml_inputs_manifest.json"

# ------------------------------------------------------------
# 1.4 Execution switches
# ------------------------------------------------------------
RUN_SPLIT_EXPORT = True      # True이면 단일 parquet를 train/val/test parquet 파일로 분할 export
COPY_FEATURE_COLUMNS = True  # True이면 FEATURE_COLUMNS_SOURCE_PATH를 FEATURE_COLUMNS_EXPORT_PATH로 복사
SAVE_MANIFEST = True         # True이면 후속 실행에 필요한 경로, feature hash, row count를 JSON으로 저장

# ------------------------------------------------------------
# 1.5 Overwrite policy
# ------------------------------------------------------------
# 기존 산출물이 있으면 실행을 중단하는 정책 ,재실행 시에는 overwrite=True보다 RUN_ID 변경 권장
OVERWRITE_POLICY = {
    "split_export": False,
    "feature_columns_copy": False,
    "manifest": False,
}
# ------------------------------------------------------------
# 1.6 Configuration preview
# ------------------------------------------------------------
print("=" * 80)
print("SOURCE_EXPERIMENT_ID        :", SOURCE_EXPERIMENT_ID)
print("EXPORT_EXPERIMENT_ID        :", EXPORT_EXPERIMENT_ID)
print("EXPORT_NAME                 :", EXPORT_NAME)
print("RUN_ID                      :", RUN_ID)
print("INPUT_SINGLE_PARQUET        :", INPUT_SINGLE_PARQUET)
print("FEATURE_COLUMNS_SOURCE_PATH :", FEATURE_COLUMNS_SOURCE_PATH)
print("EXPORT_OUTPUT_DIR           :", EXPORT_OUTPUT_DIR)
print("TRAIN_PATH                  :", TRAIN_PATH)
print("VAL_PATH                    :", VAL_PATH)
print("TEST_PATH                   :", TEST_PATH)
print("SPLIT_SUMMARY_PATH          :", SPLIT_SUMMARY_PATH)
print("SPLIT_BUILD_SUMMARY_PATH    :", SPLIT_BUILD_SUMMARY_PATH)
print("FEATURE_COLUMNS_EXPORT_PATH :", FEATURE_COLUMNS_EXPORT_PATH)
print("MANIFEST_PATH               :", MANIFEST_PATH)
print("=" * 80)

## 2. 입력 계약 검증 함수 정의

단일 parquet와 feature catalog가 후속 ML 입력 계약을 만족하는지 확인하는 함수를 정의합니다. 실제 검증은 feature catalog 사본을 만든 뒤 `FEATURE_COLUMNS_EXPORT_PATH` 기준으로 수행합니다.

In [ ]:
def require_file(path: Path, label: str) -> None:
    """
    필수 입력 파일이 실제로 존재하는지 확인
    이함수는 실행 초기에 입력 경로 오류를 빠르게 드러내기 위한 검증용
    파일이 없는데 뒤 단계까지 진행하면 parquet/schema 검증이나 copy 단계에서
    원인을 알기 어려운 에러가 날 수 있으므로 여기서 명확히 중단합니다.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found: {path}\n"
            "전처리 산출물이 생성/복사되었는지, 또는 설정 셀의 경로가 맞는지 확인하세요."
        )
        
        
def require_no_existing_file(path: Path, *, overwrite: bool, label: str) -> None:
    """
    기존 산출물을 조용히 덮어쓰지 않도록 차단
    overwrite=False이면 같은 경로에 파일이 이미 있을 때 실행을 중단
    재실행이 필요하면 overwrite=True보다 RUN_ID를 변경해 새 output directory를
    만드는 방식을 우선 권장
    """
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"Existing {label} found: {path}. "
            "Set overwrite=True or change RUN_ID to create a new output directory."
        )
        

def parquet_schema_names(path: Path) -> list[str]:
    """
    parquet 전체를 메모리에 올리지 않고 schema의 컬럼명만 반환
    여기서는 입력 계약 검증에 컬럼명만 필요하므로 schema만 read
    """
    return list(pq.ParquetFile(path).schema_arrow.names)


def parse_used_in_ml(series: pd.Series) -> pd.Series:
    """
    feature catalog의 used_in_ml 컬럼을 strict boolean mask로 변환
    허용값:
    - True 계열: true, 1, yes, y
    - False 계열: false, 0, no, n
    빈 값이나 정의되지 않은 값은 조용히 False로 처리하지 않고 에러로 중단
    used_in_ml은 실제 학습 feature 선택을 결정하므로 실패를 숨기면 안 됩니다.
    """
    if series.isna().any():
        missing_rows = (series[series.isna()].index + 2).tolist()
        raise ValueError(f"used_in_ml contains missing values. csv_rows={missing_rows[:30]}")
    if pd.api.types.is_bool_dtype(series):
        return series
    true_values = {"true", "1", "yes", "y"}
    false_values = {"false", "0", "no", "n"}
    normalized = series.astype(str).str.strip().str.lower()
    invalid = normalized[~normalized.isin(true_values | false_values)]
    if not invalid.empty:
        invalid_rows = (invalid.index + 2).tolist()
        raise ValueError(
            "used_in_ml contains unsupported values. "
            f"invalid_values={sorted(invalid.unique().tolist())[:30]}, "
            f"csv_rows={invalid_rows[:30]}"
        )
    return normalized.isin(true_values)

def load_selected_feature_columns(feature_columns_path: Path) -> list[str]:
    """feature catalog에서 used_in_ml=True인 column_name 목록을 CSV 순서대로 반환합니다.
    이 함수가 반환한 feature list가 후속 ml_train.py, ml_val.py, ml_test.py에서
    실제 모델 입력 컬럼 목록으로 사용됩니다.
    검증 항목:
    - 필수 컬럼 존재 여부: column_name, used_in_ml
    - used_in_ml 값의 strict boolean 변환 가능 여부
    - 선택된 column_name의 결측/공백 여부
    - 선택된 feature 중복 여부
    - 선택된 feature가 1개 이상인지 여부
    - label/target 등 누수 위험 컬럼명 포함 여부
    """
    feature_table = pd.read_csv(feature_columns_path, encoding="utf-8-sig")
    required_columns = {"column_name", "used_in_ml"}
    missing_columns = required_columns - set(feature_table.columns)
    if missing_columns:
        raise ValueError(
            f"Feature columns CSV is missing columns: {sorted(missing_columns)}. "
            f"path={feature_columns_path}"
        )
    used_in_ml_mask = parse_used_in_ml(feature_table["used_in_ml"])
    selected = feature_table.loc[used_in_ml_mask, "column_name"]
    if selected.isna().any():
        missing_rows = (selected[selected.isna()].index + 2).tolist()
        raise ValueError(f"Selected feature rows contain missing column_name. csv_rows={missing_rows[:30]}")
    feature_columns = selected.astype(str).str.strip().tolist()
    blank_rows = [index + 2 for index, column in zip(selected.index, feature_columns) if not column]
    if blank_rows:
        raise ValueError(f"Selected feature rows contain blank column_name. csv_rows={blank_rows[:30]}")
    duplicated = sorted({column for column in feature_columns if feature_columns.count(column) > 1})
    if duplicated:
        raise ValueError(f"Duplicated selected feature columns: {duplicated}")
    if not feature_columns:
        raise ValueError(f"No usable feature columns found. path={feature_columns_path}")
    validate_no_forbidden_feature_columns(feature_columns)
    return feature_columns

def feature_columns_hash(feature_columns: list[str]) -> str:
    """순서에 민감한 feature list fingerprint를 계산합니다.
    같은 feature 집합이라도 컬럼 순서가 바뀌면 모델 입력 행렬의 의미가 바뀔 수 있습니다.
    따라서 feature 이름뿐 아니라 순서까지 포함해 hash를 계산합니다.
    """
    payload = json.dumps(feature_columns, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def preview_input_contract(
    *,
    input_parquet_path: Path,
    feature_columns_path: Path,
    label_col: str,
) -> list[str]:
    """단일 parquet와 feature catalog가 ML 입력 export 계약을 만족하는지 확인합니다.
    이 함수는 실제 split export 전에 실행하는 사전 검증입니다.
    확인 항목:
    - 단일 parquet 파일 존재 여부
    - feature catalog CSV 파일 존재 여부
    - parquet에 필수 메타 컬럼 존재 여부: tx_id, timestamp, split, label_col
    - feature catalog에서 used_in_ml=True로 선택된 컬럼이 parquet에 모두 존재하는지 여부
    - 선택 feature 개수와 feature hash 출력
    반환값:
    - used_in_ml=True인 feature 컬럼 목록
    """
    require_file(input_parquet_path, "INPUT_SINGLE_PARQUET")
    require_file(feature_columns_path, "feature_columns_path")
    parquet_columns = parquet_schema_names(input_parquet_path)
    parquet_column_set = set(parquet_columns)
    required_meta_columns = {"tx_id", "timestamp", "split", label_col}
    missing_meta_columns = sorted(required_meta_columns - parquet_column_set)
    if missing_meta_columns:
        raise ValueError(f"single parquet is missing required meta columns: {missing_meta_columns}")
    feature_columns = load_selected_feature_columns(feature_columns_path)
    missing_features = [column for column in feature_columns if column not in parquet_column_set]
    if missing_features:
        raise ValueError(
            "used_in_ml=True feature columns are missing from input parquet. "
            f"missing={missing_features[:30]}, missing_count={len(missing_features)}"
        )
        
    print("input_parquet_path      :", input_parquet_path)
    print("feature_columns_path    :", feature_columns_path)
    print("input parquet column_count:", len(parquet_columns))
    print("selected feature_count    :", len(feature_columns))
    print("feature_columns_hash      :", feature_columns_hash(feature_columns))
    print("first_features            :", feature_columns[:10])
    print("input contract            : OK")
    return feature_columns


print("input contract helper functions loaded")

## 3. 원본 단일 parquet split 분포 확인

`split`, `label` 두 컬럼만 읽어서 원본 단일 parquet가 train/val/test를 모두 포함하는지 확인합니다.

In [ ]:
def summarize_split_labels(path: Path, *, expected_split: str | None = None) -> pd.DataFrame:
    """parquet의 split별 row 수와 label 분포를 확인합니다."""
    df = pd.read_parquet(path, columns=["split", LABEL_COL])
    df["split"] = df["split"].astype("string").str.strip().str.lower()
    observed = set(df["split"].dropna().unique().tolist())
    if expected_split is not None and observed != {expected_split}:
        raise ValueError(f"unexpected split values in {path}: expected={expected_split}, observed={sorted(observed)}")
    if expected_split is None and observed != {"train", "val", "test"}:
        raise ValueError(f"unexpected split values in {path}: {sorted(observed)}")
    rows = []
    split_order = [expected_split] if expected_split is not None else ["train", "val", "test"]
    for split_name in split_order:
        part = df[df["split"] == split_name]
        positive_count = int((part[LABEL_COL] == 1).sum())
        rows.append({
            "split": split_name,
            "rows": int(len(part)),
            "positive_count": positive_count,
            "negative_count": int(len(part) - positive_count),
            "positive_rate": None if len(part) == 0 else float(positive_count / len(part)),
            "both_classes": 0 < positive_count < len(part),
        })
    return pd.DataFrame(rows)


input_split_summary = summarize_split_labels(INPUT_SINGLE_PARQUET)
display(input_split_summary)

## 4. 단일 parquet -> ML 입력 parquet 생성

`fb_build.split_single_parquet_by_existing_split()`만 사용합니다. 이 함수는 feature를 새로 계산하지 않고 기존 `split` 컬럼을 기준으로 파일을 나눕니다.

In [ ]:
def export_split_parquets():
    """전처리 단일 parquet를 후속 ML 모듈 입력용 split parquet 3개로 분리합니다."""
    if not RUN_SPLIT_EXPORT:
        print("Split export skipped. Existing split files will be validated if present.")
        return None
    result = split_single_parquet_by_existing_split(
        input_path=INPUT_SINGLE_PARQUET,
        output_dir=EXPORT_OUTPUT_DIR,
        base_dir=PROJECT_ROOT,
        experiment_id=ARTIFACT_PREFIX,
        overwrite=OVERWRITE_POLICY["split_export"],
        tx_id_col="tx_id",
        timestamp_col="timestamp",
        label_col=LABEL_COL,
        split_col="split",
    )
    print("row_counts:", result.row_counts)
    print("train_path:", result.output_paths.train_path)
    print("val_path  :", result.output_paths.val_path)
    print("test_path :", result.output_paths.test_path)
    display(result.split_summary)
    return result


split_export_result = export_split_parquets()

In [ ]:
def prepare_feature_columns_for_ml() -> Path:
    """ML 학습에 사용할 feature catalog 사본을 준비합니다.

    원본 FEATURE_COLUMNS_SOURCE_PATH는 전처리 산출물이므로 수정하지 않습니다.
    COPY_FEATURE_COLUMNS=True이면 원본 CSV를 FEATURE_COLUMNS_EXPORT_PATH로 복사합니다.
    COPY_FEATURE_COLUMNS=False이면 사용자가 이미 준비한 FEATURE_COLUMNS_EXPORT_PATH를 그대로 사용합니다.
    """
    if COPY_FEATURE_COLUMNS:
        require_file(FEATURE_COLUMNS_SOURCE_PATH, "FEATURE_COLUMNS_SOURCE_PATH")
        require_no_existing_file(
            FEATURE_COLUMNS_EXPORT_PATH,
            overwrite=OVERWRITE_POLICY["feature_columns_copy"],
            label="feature columns export",
        )
        FEATURE_COLUMNS_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(FEATURE_COLUMNS_SOURCE_PATH, FEATURE_COLUMNS_EXPORT_PATH)
        print("feature_columns copied:", FEATURE_COLUMNS_EXPORT_PATH)
    else:
        require_file(FEATURE_COLUMNS_EXPORT_PATH, "FEATURE_COLUMNS_EXPORT_PATH")
        print("feature_columns copy skipped. Using existing export CSV:", FEATURE_COLUMNS_EXPORT_PATH)

    return FEATURE_COLUMNS_EXPORT_PATH


prepare_feature_columns_for_ml()

# ML 학습에 들어갈 최종 feature catalog는 FEATURE_COLUMNS_EXPORT_PATH입니다.
# 복사 또는 수동 수정이 끝난 뒤 이 파일 기준으로 feature 목록과 parquet schema 정합성을 검증합니다.
FEATURE_COLUMNS = preview_input_contract(
    input_parquet_path=INPUT_SINGLE_PARQUET,
    feature_columns_path=FEATURE_COLUMNS_EXPORT_PATH,
    label_col=LABEL_COL,
)

## 5. 생성 산출물 검증

생성된 split parquet 3개가 후속 ML 모듈 입력 계약을 만족하는지 확인합니다.

In [ ]:
def validate_exported_split_file(path: Path, *, expected_split: str, feature_columns: list[str]) -> pd.DataFrame:
    """후속 ml_*.py 입력으로 사용할 split parquet 하나를 검증합니다."""
    require_file(path, f"{expected_split}_parquet")
    schema_names = set(parquet_schema_names(path))
    required_columns = set(feature_columns) | {LABEL_COL, "split"}
    missing = sorted(required_columns - schema_names)
    if missing:
        raise ValueError(
            f"Exported {expected_split} parquet is missing required columns. "
            f"missing={missing[:30]}, missing_count={len(missing)}, path={path}"
        )
    return summarize_split_labels(path, expected_split=expected_split)


export_summaries = []
for split_name, split_path in {"train": TRAIN_PATH, "val": VAL_PATH, "test": TEST_PATH}.items():
    summary = validate_exported_split_file(
        split_path,
        expected_split=split_name,
        feature_columns=FEATURE_COLUMNS,
    )
    summary.insert(0, "path", str(split_path))
    export_summaries.append(summary)

export_split_summary = pd.concat(export_summaries, ignore_index=True)
display(export_split_summary)
print("export validation: OK")

## 6. Manifest 저장

후속 `ml_train.py`, `ml_val.py`, `ml_test.py`에 넘길 경로와 feature fingerprint를 JSON으로 저장합니다.

In [ ]:
def save_manifest() -> dict:
    """후속 ML 실행에 필요한 입력 경로와 검증 요약을 저장합니다."""
    manifest = {
        "notebook": "ml/ml-00_baseline/feature_build/fb_ml00_baseline_test_run.ipynb",
        "purpose": "Export ML-00 input artifacts from 03_ml_feature_process single parquet.",
        "ml_training_executed": False,
        "seed": SEED,
        "source_experiment_id": SOURCE_EXPERIMENT_ID,
        "export_experiment_id": EXPORT_EXPERIMENT_ID,
        "export_name": EXPORT_NAME,
        "run_id": RUN_ID,
        "artifact_prefix": ARTIFACT_PREFIX,
        "input_single_parquet": str(INPUT_SINGLE_PARQUET),
        "feature_columns_source": str(FEATURE_COLUMNS_SOURCE_PATH),
        "feature_columns_for_ml": str(FEATURE_COLUMNS_EXPORT_PATH),
        "outputs": {
            "train_path": str(TRAIN_PATH),
            "val_path": str(VAL_PATH),
            "test_path": str(TEST_PATH),
            "feature_columns_path": str(FEATURE_COLUMNS_EXPORT_PATH),
            "split_summary_path": str(SPLIT_SUMMARY_PATH),
            "split_existing_summary_path": str(SPLIT_BUILD_SUMMARY_PATH),
        },
        "feature_count": len(FEATURE_COLUMNS),
        "feature_columns_hash": feature_columns_hash(FEATURE_COLUMNS),
        "feature_columns": FEATURE_COLUMNS,
        "input_split_summary": input_split_summary.to_dict(orient="records"),
        "export_split_summary": export_split_summary.to_dict(orient="records"),
    }
    if SAVE_MANIFEST:
        require_no_existing_file(MANIFEST_PATH, overwrite=OVERWRITE_POLICY["manifest"], label="manifest")
        MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
        print("manifest saved:", MANIFEST_PATH)
    return manifest


ML_INPUT_MANIFEST = save_manifest()

## 7. 후속 ML 모듈 입력 경로

아래 경로를 후속 `ml_train.py`, `ml_val.py`, `ml_test.py` 실행 설정에 넣으면 됩니다. 이 노트북은 여기서 종료합니다.

In [ ]:
artifact_paths = {
    "train_path": TRAIN_PATH,
    "val_path": VAL_PATH,
    "test_path": TEST_PATH,
    "split_summary_path": SPLIT_SUMMARY_PATH,
    "split_build_summary_path": SPLIT_BUILD_SUMMARY_PATH,
    "feature_columns_path": FEATURE_COLUMNS_EXPORT_PATH,
    "manifest_path": MANIFEST_PATH,
}
artifact_status = pd.DataFrame(
    [{"name": name, "exists": path.exists(), "path": str(path)} for name, path in artifact_paths.items()]
)
display(artifact_status)

print("후속 ml_train.py 입력 예시")
print("train_path           =", TRAIN_PATH)
print("val_path             =", VAL_PATH)
print("feature_columns_path =", FEATURE_COLUMNS_EXPORT_PATH)
print()
print("후속 ml_val.py 입력 예시")
print("val_path             =", VAL_PATH)
print()
print("후속 ml_test.py 입력 예시")
print("test_path            =", TEST_PATH)
print()
print("추적용 export 산출물")
print("split_summary_path       =", SPLIT_SUMMARY_PATH)
print("split_build_summary_path =", SPLIT_BUILD_SUMMARY_PATH)
print("manifest_path            =", MANIFEST_PATH)
print()
print("주의: 이 노트북은 학습/validation/test를 실행하지 않았습니다.")